# பாடம் 03 - முகவர் வடிவமைப்பு மாதிரிகள்

இந்த பாடத்தில், பயனுள்ள AI முகவர்களை உருவாக்க மூன்று அடிப்படைக் கட்டமைப்பு மாதிரிகளைப் பார்க்கிறோம்:

1. **தெளிவான முகவர் கொள்கைகள்** — முகவர் நடத்தையை வழிநடத்திய தெளிவான, பங்கு வரையறுக்கும் பிரதிகள் உருவாக்குதல்
2. **Pydantic மாதிரிகளுடன் கட்டமைக்கப்பட்ட வெளியீடு** — முகவர்கள் கணிக்கப்பட்ட, சரிபார்க்கப்பட்ட தரவை வழங்குவதை உறுதிப்படுத்தல்
3. **ஒற்றை பொறுப்பு முகவர்கள்** — ஒவ்வொரு முகவரும் ஒரு செயல்பாட்டை நன்கு செய்ய வடிவமைத்தல்

நாம் ஒவ்வொரு மாதிரியையும் **பயண இலக்கு பரிந்துரைசெய்யும்** சூழலில் செயல்படுத்தி, இலக்குகளை பரிந்துரைக்கும், கிடைக்கும்தைக் காண்பிக்கும், மற்றும் பொருட்கள் நடத்தலை கையாளும் ஒரு அமைப்பை படிப்படியாக உருவாக்குவோம்.


## அமைப்பு


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic python-dotenv --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## மாதிரி 1: தெளிவான முகவர் வழிகாட்டிகள்

மிகப் பெரிய தாக்கத்தை கொடுக்கக்கூடிய மாதிரியும் மிகவும் எளிமையானதுமானது: உங்கள் முகவருக்கான தெளிவான, விரிவான வழிகாட்டிகளை எழுதுவது.

நல்ல வழிகாட்டிகள் வரையறுக்கும் விஷயங்கள்:
- **யார்** முகவர் (பாத்திரம் மற்றும் ஸ்வரு)
- **என்ன** அது செய்யவேண்டும் (படி படியாக உள்ள பொறுப்புகள்)
- **எப்படி** அது நடந்து கொள்ளவேண்டும் (குறைப்பாடுகள் மற்றும் பாணி)

கீழே, நாம் ஒரு பயண இணைப்பாளர் முகவரைக் உருவாக்குகிறோம், அவன் ஒவ்வொரு பதிலையும் உருவாக்கும் போது தெளிவான வழிகாட்டிகளை பயன்படுத்துகிறோம்.


In [ ]:
agent = provider.as_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## மாதிரி 2: பைடான்டிக் மாடல்களுடன் கட்டமைக்கப்பட்ட வெளியீடு

சுதந்திர வடிவில் உள்ள உரை உரையாடலுக்கு பயனுள்ளது, ஆனால் கீழ்தள் அமைப்புகளுக்கு கட்டமைக்கப்பட்ட தரவு அவசியம்.
**பைடான்டிக் மாடல்களை** ஒரு **கருவி செயல்பாட்டுடன்** இணைத்து, நாம் முடியும்:

- முகவரின் வெளியீட்டிற்கு ஒரு சரியான வடிவமைப்பை நிர்ணயிக்க
- பதிலை தானாக சோதனை செய்ய
- முகவர் முடிவுகளை பயன்பாட்டு தர்க்கத்தில் நம்பகமாக இணைக்க

கட்டாயப்படுத்துவதற்கான முக்கியம் முகவரை இயக்கும் போது `response_format` ஐ வழங்குவதை விடுவதுதான். இது
மாடலை ஒரு சோதிக்கப்பட்ட `TravelRecommendations` பொருளை ( `response.value` இல் கிடைக்கிறது )
சுதந்திர உரையின் பதிலாக வழங்கும் வகையில் கட்டாயப்படுத்தும். `get_destination_details` கருவியும் ஒரு வகைப்பட்ட
`DestinationRecommendation` ஐ வழங்கும், ஆகவே தரவு முடிவில் கட்டமைக்கப்பட்டே இருக்கும்.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(
    destination: Annotated[str, "The destination to look up"]
) -> DestinationRecommendation:
    """Get structured details about a vacation destination."""
    details = {
        "Barcelona": DestinationRecommendation(
            destination="Barcelona",
            available=True,
            best_season="May-Jun",
            highlights=["Beach", "Architecture", "Nightlife"],
            estimated_budget_usd=2000,
        ),
        "Tokyo": DestinationRecommendation(
            destination="Tokyo",
            available=True,
            best_season="Mar-Apr",
            highlights=["Culture", "Food", "Technology"],
            estimated_budget_usd=2500,
        ),
        "Cape Town": DestinationRecommendation(
            destination="Cape Town",
            available=False,
            best_season="Nov-Mar",
            highlights=["Nature", "Wine", "Adventure"],
            estimated_budget_usd=1800,
        ),
    }
    return details.get(
        destination,
        DestinationRecommendation(
            destination=destination,
            available=False,
            best_season="Unknown",
            highlights=[],
            estimated_budget_usd=0,
        ),
    )


structured_agent = provider.as_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

# Passing `response_format` forces the agent to return a validated
# TravelRecommendations object instead of free-form text.
response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget",
    options={"response_format": TravelRecommendations},
)

if response and response.value:
    result: TravelRecommendations = response.value
    for rec in result.recommendations:
        status = "Available" if rec.available else "Not available"
        print(f"{rec.destination} ({status})")
        print(f"  Best season: {rec.best_season}")
        print(f"  Highlights: {', '.join(rec.highlights)}")
        print(f"  Estimated budget: ${rec.estimated_budget_usd}")
        print()
    print(f"Note: {result.personalized_note}")
else:
    print("No validated structured response was returned.")
    print(response)


## முறையமைப்பு 3: ஒற்றை பொறுப்பு முகவர்கள்

சிக்கலான பணிகள் பல கவனச்சிதறல் இல்லாத முகவர்களிடையே வேலை பகிர்வதால் பலன் ஏற்படும், ஒவ்வொரு முகவரும் ஒரே பொறுப்பை ஏற்றுக் கொள்கின்றனர்:

- இடங்கள் மற்றும் கிடைக்கும் நிலை பற்றி அறிந்த **இடமாற்ற நிபுணர்**
- பறப்புகள், ஹோட்டல்கள் மற்றும் பயணத் திட்டங்களை கையாளும் **சரக்குத்துறைக் திட்டமிடுபவர்**

இது மென்பொருள் பொறியியல் கோட்பாடு *கவலை பகிர்வு* னை பிரதிபலிக்கிறது—ஒவ்வொரு முகவரும் தனித்து சோதனை செய்ய, பராமரிக்க மற்றும் மேம்படுத்த எளிதாகும்.


In [ ]:
destination_agent = provider.as_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = provider.as_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## சுருக்கம்

இந்த படிப்பில் நாம் ஒரு பயண பரிந்துரையாளர் சூழ்நிலைக்கு மூன்று முகவரி வடிவமைப்பு முறைகள் பயன்படுத்தினோம்:

| வடிவமைப்பு | முக்கியக் கருத்து | நன்மை |
|---|---|---|
| **தெளிவான அறிவுறுத்தல்கள்** | முன்னதாகவே நபர், பொறுப்புகள் மற்றும் கட்டுப்பாடுகளை வரையறுக்கவும் | ஒரே மாதிரியில், பிராண்டிற்கு ஏற்ப அடையாள முகவரி நடத்தை |
| **கட்டமைக்கப்பட்ட வெளியீடு** | பதில் வடிவமாக Pydantic மாதிரிகளை பயன்படுத்தவும் | சரிபார்க்கப்பட்ட, இயந்திர படிக்கக் கூடிய முடிவுகள் |
| **ஒரே பொறுப்பு** | ஒவ்வொரு முகவரிக்கும் ஒரு கவனிக்கப்பட்ட வேலை அளிக்கவும் | சுலபமாக சோதிக்க, பராமரிக்க மற்றும் சேர்க்கக்கூடியது |

இந்த வடிவமைப்புகள் இயல்பாக கூடியவை — தெளிவான அறிவுறுத்தல்களை கட்டமைக்கப்பட்ட வெளியீடு உடன் ஒரே பொறுப்புள்ள முகவரியில் சேர்த்து வலுவான, தயாரிப்பு முறைமை உருவாக்கலாம்.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**மறுப்பு**:
இந்த ஆவணம் AI மொழிபெயர்ப்பு சேவை [Co-op Translator](https://github.com/Azure/co-op-translator) பயன்படுத்தி மொழிபெயர்க்கப்பட்டுள்ளது. நாங்கள் துல்லியத்திற்காக முயற்சி செய்துள்ளோம், ஆனால் தானாக செய்யப்படும் மொழிபெயர்ப்புகளில் பிழைகள் அல்லது தவறுகள் இருக்கலாம் என்பதை கவனத்தில் கொள்ளவும். அசல் ஆவணம் அதன் தாய்மொழியில் அதிகாரப்பூர்வ ஆதாரமாக கருதப்பட வேண்டும். முக்கியமான தகவல்களுக்கு, தொழில்நுட்பமான மனித மொழிபெயர்ப்பு பரிந்துரைக்கப்படுகிறது. இந்த மொழிபெயர்ப்பைப் பயன்படுத்துவதால் ஏற்படும் எந்த தவறான புரிதல்கள் அல்லது தவறான விளக்கத்திற்கும் நாங்கள் பொறுப்பில்வில்லை.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
